# Train, Tune, and Assess Thin-Cloud Classifiers

This notebook runs the structured training pipeline for the three final models: Decision Tree, Random Forest, and XGBoost. The reusable logic lives in `src/lswt_cloud_masking`; this notebook is only for configuration, execution, and quick inspection of outputs.

In [9]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from lswt_cloud_masking.model_training import TrainingConfig, run_training_pipeline

## Configure

The default paths mirror the old merged train/test folder. Adjust `config` below if your CSVs or output folder are elsewhere.

In [10]:
config = TrainingConfig.from_json(ROOT / "configs" / "training_config.example.json")
# Resolve config paths relative to the repository root
config.train_csv = str((ROOT / config.train_csv).resolve())
config.test_csv = str((ROOT / config.test_csv).resolve())
config.output_dir = str((ROOT / config.output_dir).resolve())

print("Train exists:", Path(config.train_csv).exists(), config.train_csv)
print("Test exists:", Path(config.test_csv).exists(), config.test_csv)
print("Output directory:", config.output_dir)


# For a smoke test, uncomment these three lines before running the full Optuna search.
# config.n_trials_dt = 2
# config.n_trials_rf = 2
# config.n_trials_xgb = 2

config

Train exists: True /Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/data/df_train_all.csv
Test exists: True /Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/data/df_test_all.csv
Output directory: /Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/models/general


TrainingConfig(train_csv='/Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/data/df_train_all.csv', test_csv='/Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/data/df_test_all.csv', output_dir='/Users/airahaghi/Documents/Trishna/Scripts/LSWT-ML-thin-cloud-masking/models/general', label_column='lst_class', drop_columns=['lakeN', 'lst_raw', 'lst_filt', 'raamap', 'vzamap', 'szamap'], random_state=42, cv_splits=5, scoring='balanced_accuracy', n_trials_dt=80, n_trials_rf=120, n_trials_xgb=120, optuna_n_jobs=1)

## Run Training

This writes tuned models, Optuna studies, reports, confusion matrices, and metadata under `config.output_dir`.

In [11]:
result = run_training_pipeline(config)
result.keys()

[I 2026-08-05 09:52:32,309] A new study created in memory with name: DecisionTree_Optimization
[I 2026-08-05 09:52:33,517] Trial 0 finished with value: 0.7031839031460883 and parameters: {'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 10, 'criterion': 'log_loss', 'splitter': 'best'}. Best is trial 0 with value: 0.7031839031460883.
[I 2026-08-05 09:52:33,589] Trial 1 finished with value: 0.6187454687686209 and parameters: {'max_depth': 11, 'min_samples_split': 19, 'min_samples_leaf': 9, 'criterion': 'log_loss', 'splitter': 'random'}. Best is trial 0 with value: 0.7031839031460883.
[I 2026-08-05 09:52:33,636] Trial 2 finished with value: 0.5795970142267582 and parameters: {'max_depth': 7, 'min_samples_split': 8, 'min_samples_leaf': 1, 'criterion': 'gini', 'splitter': 'random'}. Best is trial 0 with value: 0.7031839031460883.
[I 2026-08-05 09:52:34,370] Trial 3 finished with value: 0.6055666732551759 and parameters: {'max_depth': 8, 'min_samples_split': 19, 'min_samples_leaf

dict_keys(['models', 'studies', 'metrics', 'metadata'])

In [12]:
import pandas as pd

summary = pd.DataFrame({
    name: {
        "test_accuracy": metrics["test_accuracy"],
        "test_balanced_accuracy": metrics["test_balanced_accuracy"],
        "cv_mean": metrics["cv_mean"],
        "cv_std": metrics["cv_std"],
    }
    for name, metrics in result["metrics"].items()
}).T
summary

,test_accuracy,test_balanced_accuracy,cv_mean,cv_std
decision_tree,0.752362,0.751472,0.737032,0.005052
random_forest,0.871457,0.870810,0.857411,0.007150
xgboost,0.886220,0.885687,0.877202,0.003437


The saved model paths are recorded in `model_metadata.json`. Use the RF and XGBoost paths in the single-scene and batch masking pipelines.